# Collective motion of zebrafish

What this notebook does, in order:

1. **Writes a track converter** for a format mosaic does not ship -- the
   Ctrax/JAABA `trx` MATLAB struct. This tracker reports a centroid, an
   orientation and a fitted ellipse, and **locates no keypoints at all**, so it
   is a useful case for anyone whose own tracker does the same: the table it
   produces carries `X`, `Y` and a heading, and no pose columns.
2. **Measures collective motion**: the polarization and rotation order
   parameters of Tunstrom et al., and the collective state they define.
3. **Builds social force maps**: how a fish turns and changes its speed
   depending on where its nearest neighbor is, in its own body frame. That is
   the `nearest-neighbor` -> `nn-delta-response` -> `nn-delta-bins` chain.

The notebook is in two parts. **Part A** runs the pipeline: every call into
mosaic lives there, and it writes results to disk. **Part B** loads those
results and plots them, with no mosaic calls at all. The split is deliberate --
it shows which of this is the toolkit and which is ordinary matplotlib you would
write yourself.

## Getting the data: two ways

`SOURCE` in the next cell picks one.

**`"download"` (the default)** fetches the converted tables from
[EcodylicScience/mosaic-example-zebrafish][card] on Hugging Face -- 767 MB --
together with a 79 MB two-trial raw sample. Every cell below still
runs: the sample is real `.mat` output, the converter written in section 3 is
executed on it, and its output is checked against the shipped tables before the
analysis begins. So the conversion is demonstrated rather than described.

**`"convert"`** builds all 33 tables from the Dryad download. That is the path
to copy for your own recordings, and the only one that reproduces every table
from scratch.

Either way, sections 6 onwards analyse all 33 trials.

## The data

Wild-type (AB strain) zebrafish, six per group, free-swimming in a circular
arena about 48 cm across, filmed from above at 60 Hz for 30 minutes.
Thirty-three trials, about 21.4 million rows once converted.

**Citation.** Tang W, Davidson JD, Zhang G, Conen KE, Fang J, Serluca F, Li J,
Xiong X, Coble M, Tsai T, Molind G, Fawcett CH, Sanchez E, Zhu P, Couzin ID,
Fishman MC (2020) Genetic Control of Collective Behavior in Zebrafish.
*iScience* 23(3): 100942.
[doi:10.1016/j.isci.2020.100942](https://doi.org/10.1016/j.isci.2020.100942)

**Data licence.** CC0 1.0, from Dryad:
[doi:10.5061/dryad.hx3ffbg9n](https://doi.org/10.5061/dryad.hx3ffbg9n). The
converted tables on Hugging Face are a derivative of that record and carry the
same licence. To build them yourself, download the record and set
`DATA_DIR` below to the extracted wild-type directory -- the one holding one
`.mat` per trial.

[card]: https://huggingface.co/datasets/EcodylicScience/mosaic-example-zebrafish

In [ ]:
import re
import shutil
import tarfile
import tempfile
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat

# ---- configuration ---------------------------------------------------------
# Where the tracks come from.
#
#   "download" -- fetch the converted tables from Hugging Face (~900 MB), plus a
#                 two-trial raw sample so every cell below still runs against
#                 real .mat files. This is what most people want.
#   "convert"  -- build the tables yourself from the Dryad download. Set
#                 DATA_DIR to the extracted wild-type directory.
#
# The converter is written out and executed in both modes. In "download" it runs
# on the sample and its output is checked against the tables that were shipped,
# so nothing below is taken on trust.
SOURCE = "download"

# ---- what this costs on disk, in either mode -------------------------------
# Sections 6-9 run over all 33 trials whichever mode you are in, so the feature
# output is the same either way: about 9 GB, most of it nn-delta-response. On top
# of that, download mode fetches ~846 MB and extracts ~800 MB, and huggingface_hub
# keeps the archives in its own cache as well.

# -- download mode --
HF_REPO = "EcodylicScience/mosaic-example-zebrafish"
# Where the downloaded dataset is unpacked. None -> ./zebrafish-example
DOWNLOAD_DIR: Optional[Path] = None

# -- convert mode --
# The extracted wild-type directory, holding one .mat per trial.
DATA_DIR: Optional[Path] = None
# Where the dataset is built. None -> a temp directory. See the disk note above,
# and point this somewhere with room.
PROJECT_DIR: Optional[Path] = None

# True re-converts and recomputes everything. Left False a re-run costs minutes:
# conversion skips tables that already exist, and every feature run is
# content-addressed, so identical params and inputs resolve to the same run_id
# and nothing is recomputed.
RESET = False

FPS = 60.0

# How far ahead to look for the response in the force maps, in frames.
# 20 frames at 60 Hz is 333 ms.
DIFF_NUMFRAMES = 20

# Frames to drop from the start of each trial before the force maps. None keeps
# everything; nothing in these files records when the fish were introduced.
ACCLIMATION_FRAMES: Optional[int] = None

# Processes for the frame-by-frame features. They hold the GIL, so threads buy
# nothing here. Raise it if you have the cores and the memory.
WORKERS = 2

# ---- resolve the two paths every later cell uses ---------------------------
# SRC  -- a directory of .mat files to read and convert (two, or all 33)
# WORK -- the mosaic dataset the analysis reads
if SOURCE == "download":
    try:
        from huggingface_hub import hf_hub_download
    except ImportError as exc:
        raise ImportError(
            "SOURCE='download' needs huggingface_hub:\n"
            "    pip install 'huggingface_hub>=1.2.0'\n"
            "The version floor matters: older clients retry a rate-limit "
            "response with a 25-second backoff against a 5-minute window, so "
            "they fail rather than wait. Set SOURCE='convert' to skip the "
            "download entirely and build from the Dryad tree."
        ) from exc

    root = Path(DOWNLOAD_DIR) if DOWNLOAD_DIR else Path.cwd() / "zebrafish-example"
    root.mkdir(parents=True, exist_ok=True)
    for asset, target in (
        ("zebrafish-wt-tracks.tar.gz", "zebrafish-wt-tracks"),
        ("zebrafish-raw-sample.tar.gz", "zebrafish-raw-sample"),
    ):
        if (root / target).exists():
            print(f"have      {root / target}")
            continue
        print(f"fetching  {asset} ...")
        archive = hf_hub_download(HF_REPO, asset, repo_type="dataset")
        with tarfile.open(archive) as tar:
            tar.extractall(root)
    SRC = root / "zebrafish-raw-sample"
    WORK = root / "zebrafish-wt-tracks"

elif SOURCE == "convert":
    if DATA_DIR is None:
        raise ValueError(
            "SOURCE='convert' needs DATA_DIR set to the extracted wild-type "
            "directory -- the one holding one .mat file per trial.\n"
            "\n"
            "The data is on Dryad at https://doi.org/10.5061/dryad.hx3ffbg9n\n"
            "\n"
            "Set SOURCE='download' instead to fetch the converted tables with a\n"
            "two-trial raw sample, which runs every cell below."
        )
    SRC = Path(DATA_DIR)
    WORK = Path(PROJECT_DIR) if PROJECT_DIR else Path(tempfile.mkdtemp(prefix="zfish-"))

else:
    raise ValueError(f"SOURCE must be 'download' or 'convert', not {SOURCE!r}")

WORK.mkdir(parents=True, exist_ok=True)

TRIALS = sorted(p for p in SRC.glob("*.mat") if not p.name.startswith("._"))
if not TRIALS:
    raise FileNotFoundError(f"no .mat files under {SRC}")

print(f"mode      {SOURCE}")
print(f"{len(TRIALS)} raw trial(s) in {SRC}")
print(f"work dir  {WORK}  ({shutil.disk_usage(WORK).free / 2**30:.0f} GB free)")
if SOURCE == "download":
    print(
        "\nThat is the sample. The converted tables for all 33 trials came "
        "down with it;\nsections 1-5 work through the sample, sections 6 "
        "onwards analyse all 33."
    )

# Part A. Running the pipeline

## 1. What is in the file

One `.mat` per trial, MATLAB **v5** -- read these with `scipy.io.loadmat`, not
`h5py`. Each file holds two top-level arrays and a `trx` struct array of shape
`(1, n_fish)`, one element per individual. Every per-frame field inside an
element is `(1, n_frames)`, so a whole trial reads as `(n_fish, n_frames)`:

```python
loadval = lambda k: np.array([t[0][0] for t in np.array(mat["trx"][k]).T])
```

| Field | What it is |
| --- | --- |
| `x`, `y` | the **centroid** in video pixels, image frame with +Y down |
| `theta` | body orientation in radians, head-resolved, partly unwrapped to +-1.5 pi |
| `a`, `b` | fitted ellipse semi-axes in pixels |
| `id` | 0 to 5, stable for the whole trial |
| `nframes`, `timestamps` | frame count and per-frame time in seconds |
| `fps`, `pxpermm` | 60, and 4.02361434 -- so 40.2361434 px per cm |

Two properties of this format shape the converter in section 3:

**Missing positions and missing headings are both NaN, and independent of each
other** -- a row can carry a good position and no heading. There is no
confidence or probability field.

**A failed ellipse fit is reported as a zero, not a NaN.** `a` and `b` come back
exactly `0` while `x`, `y` and `theta` are good -- on 21% of rows overall, and
anywhere from 12% to 46% depending on the trial, so it is worth printing per
trial rather than assuming a rate. Read literally a zero is a fish zero pixels
long: it pulls the median body length down and collapses the two pose keypoints
onto the centroid. The converter turns it into NaN.

### Fields this converter does not use

| Field | Why |
| --- | --- |
| `arena` | Reports center (508, 507) radius 254, while the fish occupy x 108-1983, y 97-1933. Stale fly-bowl template data. |
| `moviename`, `annname` | Paths to a *Drosophila* fly-bowl rig, identical in every file of every line. They name no real movie. |
| `x_mm`, `y_mm` | A registered millimeter frame with a mirrored x axis, about a different origin than `arena` claims. Two inconsistent frames in one file. |
| `theta_mm` | Held constant in blocks of five frames -- an effective 12 Hz on a 60 Hz recording. |
| `dt` | Reads 0.033; the true interval is 0.01666574 s. Wrong by a factor of two, and one element short. |
| `sex` | Stored per frame and constant per fish. A Ctrax artifact. |

The general rule, worth carrying to your own data: **where a tracker's metadata
disagrees with its trajectories, trust the trajectories.** The arena, the frame
rate and the body length here are all measured back out of the tracks.

In [ ]:
def loadtrx(path):
    """Every field of one trial, as (n_fish, n_frames) arrays."""
    mat = loadmat(str(path))
    trx = mat["trx"]
    out = {
        k: np.array([t[0][0] for t in np.array(trx[k]).T])
        for k in ("x", "y", "theta", "a", "b")
    }
    out["ids"] = np.asarray([t[0][0] for t in np.array(trx["id"]).T]).ravel()
    out["timestamps"] = np.asarray(mat["timestamps"], dtype=float).ravel()
    out["pxpermm"] = float(np.asarray(trx["pxpermm"].ravel()[0]).ravel()[0])
    return out


print(
    f"{'trial':28s} {'fish':>4s} {'frames':>8s} {'dur s':>7s} {'dt':>10s} "
    f"{'px/cm':>8s} {'nan xy':>7s} {'nan th':>7s} {'a=0':>6s}"
)
survey = []
for p in TRIALS:
    d = loadtrx(p)
    ts = d["timestamps"]
    row = dict(
        trial=p.stem,
        fish=d["x"].shape[0],
        frames=d["x"].shape[1],
        duration=float(ts[-1]),
        dt=float(np.median(np.diff(ts))),
        px_per_cm=10.0 * d["pxpermm"],
        nan_xy=float(np.isnan(d["x"]).mean()),
        nan_th=float(np.isnan(d["theta"]).mean()),
        zero_a=float((d["a"] == 0).mean()),
    )
    survey.append(row)
    print(
        f"{row['trial'][:28]:28s} {row['fish']:>4d} {row['frames']:>8,} "
        f"{row['duration']:>7.1f} {row['dt']:>10.8f} {row['px_per_cm']:>8.4f} "
        f"{100 * row['nan_xy']:>6.3f}% {100 * row['nan_th']:>6.3f}% "
        f"{100 * row['zero_a']:>5.1f}%"
    )

survey = pd.DataFrame(survey)
PX_PER_CM = float(survey.px_per_cm.iloc[0])
assert survey.px_per_cm.nunique() == 1, "calibration differs between trials"
assert survey.fish.nunique() == 1, "group size differs between trials"
print(
    f"\nframe count {survey.frames.min():,} to {survey.frames.max():,}, so read it "
    "per file rather than hardcoding it"
)
print(f"median dt {survey.dt.median():.8f} s = {1 / survey.dt.median():.4f} Hz")
print(f"calibration {PX_PER_CM:.4f} px/cm, identical in all {len(survey)} trials")

## 2. The track schema

A converter turns one raw file into one standard table: per frame, per
individual, with `frame, time, id, group, sequence, X, Y`. Two rules of the
`mosaic_v1` schema decide what this converter emits.

**Derived quantities are forbidden.** `mosaic_v1` refuses fifteen columns --
`VX, VY, AX, AY, SPEED, ANGLE, ANGULAR_V, X#wcentroid` and their variants --
because they belong to features, where the method is chosen and recorded, not to
converters.

`ANGLE` is on that list and this tracker measures it directly: `theta` comes
from the ellipse fit, and it agrees with the direction of travel to a median of
under a degree on moving frames. When your tracker genuinely measures a
forbidden column, declare a schema that `extends="mosaic_v1"` and `allows`
exactly that column. That is what the next cell does.

**Keypoints are optional, and this tracker has none.** It reports a centroid, an
orientation and a fitted ellipse -- it never locates a snout or a tail. So this
converter writes no `poseX*`/`poseY*` columns at all, which is the honest
answer and now the supported one.

Two temptations to avoid if your own tracker is centroid-only:

- Copying `X`/`Y` into `poseX0`/`poseY0` claims a keypoint that was never
  located, and hands a fictional landmark to anything that reads keypoints.
- Synthesizing a body axis from a fitted ellipse looks more defensible, since
  the numbers are all measured, but it inherits whatever the fit did. In these
  files the ellipse is missing on 21% of rows and implausible on another 7%
  (body lengths past 60 cm, in a 47 cm arena), so the synthetic keypoints would
  be absent or nonsense on roughly a quarter of the data.

Features that genuinely need keypoints refuse when they are absent, which is the
behavior you want: a clear failure rather than a plausible number. Nothing in
this notebook needs them -- collective motion, nearest neighbor and the force
maps all read `X`, `Y` and `ANGLE`.

In [ ]:
from mosaic.core.schema import (
    TRACK_SCHEMAS,
    TrackSchema,
    register_track_schema,
    schema_family,
)

register_track_schema(
    TrackSchema(
        name="zfish_trx_v1",
        extends="mosaic_v1",
        allows={"ANGLE"},
        description=(
            "mosaic_v1 plus the body orientation this Ctrax/JAABA-style tracker "
            "measures. It fits an ellipse per individual per frame and reports a "
            "head-resolved orientation; ANGLE is that angle, not an inference "
            "from displacement. Nothing else in the derived set is allowed: "
            "these files carry no speed, no velocity and no weighted centroid. "
            "X/Y is the tracker's centroid, as mosaic_v1 asks, and there are no "
            "keypoint columns because the tracker locates no keypoints."
        ),
    )
)

s = TRACK_SCHEMAS["zfish_trx_v1"]
print(
    "still forbidden:",
    len(s.forbidden),
    "columns, including SPEED:",
    "SPEED" in s.forbidden,
)
print("ANGLE allowed  :", "ANGLE" not in s.forbidden)
print(
    "family         :",
    schema_family("zfish_trx_v1"),
    "-> mixes with any mosaic_v1 table",
)

Two things to know about `allows`:

- It only takes effect together with `extends`. A schema declaring `forbidden=`
  and `allows=` with no `extends` registers without complaint and still refuses
  the column.
- Registering the schema in a notebook means its name is recorded in
  `tracks/index.csv` but is unknown to any other process. Conversion has to
  happen here, and it does. Feature runs in a fresh process are fine;
  `mosaic convert-tracks` from the command line is not. A converter shared
  across a team belongs in a module everyone imports -- see
  [Adding a converter](../docs/guides/tracking/write-a-converter.md).

## 3. The converter

In [ ]:
from mosaic.core.track_converter import (
    TrackConverter,
    TrackConvertParams,
    register_track_converter,
)


class ZfishTrxParams(TrackConvertParams):
    """Everything that determines the output, and nothing else.

    Entry identity travels in `EntryHints`, never here: params are hashed into
    the tracks variant id, so a sequence name in params would mint one variant
    per file where there is one recipe.
    """

    drop_undetected: bool = True
    zero_ellipse_is_missing: bool = True
    wrap_theta: bool = True
    ellipse_scale: float = 1.0
    emit_calibration: bool = True


@register_track_converter
class ZfishTrxConverter(TrackConverter[ZfishTrxParams]):
    """A Ctrax/JAABA-style `trx` .mat -> a `zfish_trx_v1` table.

    `X` / `Y` is the tracker's centroid, which is what `mosaic_v1` asks for.

    **No keypoint columns are written.** This tracker fits an ellipse; it never
    locates a landmark. `body_length` and `body_width` carry the ellipse axes as
    plain measurements, which is what they are -- but see `ellipse_scale`, and
    note that the fit fails outright on about a fifth of rows and returns
    something implausible on several percent more, so treat a per-row value with
    suspicion and prefer the median.

    Params:
        drop_undetected: Drop rows with no position. Filters on `X`/`Y` only, so
            a row whose `theta` alone is NaN keeps its position --
            `collective-motion-metrics` reports `n_ids_heading` separately from
            `n_ids`, so there is no need to lose a position to a missing heading.
        zero_ellipse_is_missing: Treat `a = b = 0` as a failed fit and write NaN
            into `body_length` / `body_width`. 21% of rows in this dataset, and
            up to 46% in the worst trial. Left as zero it reads as a real
            measurement of zero length.
        wrap_theta: Wrap `theta` to (-pi, pi]. The files carry it partly
            unwrapped, out to +-1.5 pi.
        ellipse_scale: How to read `a`. The `trx` convention is ambiguous between
            the semi-major axis (body length `2a`, about 2.2 cm here) and Ctrax's
            quarter-major axis (`4a`, about 4.4 cm), and nothing in the file
            settles it. `1.0` takes the first reading; the neighbor distances in
            section 12 support it, since `4a` would put a typical pair half a
            body length apart, which is overlapping.
        emit_calibration: Write `px_per_cm` as a column, from the file's own
            `pxpermm`. mosaic's usual home for a physical scale is the media
            index plus the `scale-to-cm` feature; there is no media here, so the
            factor travels with the table instead.

    `time` comes from the file's own `timestamps`, not from a frame rate. Frames
    are numbered from 0 here and from 1 by the tracker (`firstframe`), so frame k
    in this table is the tracker's frame k+1.
    """

    src_format = "zfish_trx_mat"
    # 0.1 also wrote a pose pair synthesized from the fitted ellipse. The
    # version is part of the tracks variant id, so bumping it is what makes an
    # existing dataset re-convert rather than silently keeping the old tables:
    # the params did not change, so the hash alone would not have moved.
    version = "0.2"
    enumerable = False  # one .mat is one trial
    merges_per_sequence = False  # ... and one trial is one sequence
    output_schema = "zfish_trx_v1"
    Params = ZfishTrxParams

    def convert(self, path, params, hints):
        mat = loadmat(str(path))
        trx = mat["trx"]
        field = lambda k: np.array([t[0][0] for t in np.array(trx[k]).T])  # noqa: E731

        x, y = field("x").astype(float), field("y").astype(float)
        theta = field("theta").astype(float)
        a, b = field("a").astype(float), field("b").astype(float)
        ids = (
            np.asarray([t[0][0] for t in np.array(trx["id"]).T])
            .ravel()
            .astype(np.int64)
        )
        n_ids, n_frames = x.shape

        timestamps = np.asarray(mat["timestamps"], dtype=float).ravel()
        if timestamps.size != n_frames:
            raise ValueError(
                f"{path.name}: {timestamps.size} timestamps for {n_frames} frames. "
                "Everything else is shaped from the trajectory arrays, so this is "
                "the one place a malformed file is caught."
            )

        if params.wrap_theta:
            theta = np.arctan2(np.sin(theta), np.cos(theta))
        if params.zero_ellipse_is_missing:
            no_fit = (a <= 0) | (b <= 0)
            a, b = np.where(no_fit, np.nan, a), np.where(no_fit, np.nan, b)

        frames = np.tile(np.arange(n_frames, dtype=np.int64), n_ids)

        columns = {
            "frame": frames,
            "time": np.tile(timestamps, n_ids),
            "id": np.repeat(ids, n_frames),
            "X": x.reshape(-1),
            "Y": y.reshape(-1),
            "ANGLE": theta.reshape(-1),
            "body_length": (2.0 * a * params.ellipse_scale).reshape(-1),
            "body_width": (2.0 * b * params.ellipse_scale).reshape(-1),
            "group": np.full(frames.size, hints.group),
            "sequence": np.full(frames.size, hints.sequence or path.stem),
        }
        if params.emit_calibration:
            columns["px_per_cm"] = np.full(
                frames.size, 10.0 * float(np.asarray(field("pxpermm")).ravel()[0])
            )

        table = pd.DataFrame(columns)
        if params.drop_undetected:
            keep = np.isfinite(table.X.to_numpy()) & np.isfinite(table.Y.to_numpy())
            table = table[keep].reset_index(drop=True)
        return table


print("registered as:", ZfishTrxConverter.src_format)

## 4. Build the dataset and convert

One `.mat` is one trial is one sequence, which is the default case: leave
`enumerable` and `merges_per_sequence` at `False` and do not override
`sequence_from_stem`. `group` is left empty -- thirty-three unique trial names
need no disambiguating namespace, and `group` is not the way to categorize
sequences for analysis (tags are).

`strict_schema=True` turns a missing required column from a printed report into
a raise. It is tagged `HASH_EXCLUDE`, so it does not mint a second tracks
variant.

Note the two file counts the cell prints. On a macOS volume every `.mat` has an
AppleDouble sidecar named `._<name>.mat` beside it, so a naive `rglob` finds
twice as many files as there are trials. mosaic's scan drops them before
`exclude_patterns` is consulted, so `patterns` needs no defensive spelling.

In [ ]:
from mosaic.core.dataset import Dataset, new_dataset_manifest
from mosaic.core.manifest import TracksScanSource, default_roots
from mosaic.core.pipeline.tracks_index import read_tracks_index


def make_dataset(path, name):
    """Load the dataset at *path*, creating a manifest only if there is none.

    In download mode the manifest arrives in the tarball and must be kept; in
    convert mode there is nothing there yet and one is written.
    """
    manifest = path / "dataset.yaml"
    return Dataset(
        manifest
        if manifest.exists()
        else new_dataset_manifest(name=name, base_dir=path, roots=dict(default_roots))
    ).load()


def convert_into(ds, src_dir):
    """Point a dataset at a directory of .mat files and convert all of them."""
    # `add_scan_source` raises on a duplicate id, and a source declaration is saved
    # into dataset.yaml -- so an unguarded call kills every re-run against a
    # persistent PROJECT_DIR, which is exactly what the prose above recommends.
    # Redeclare only when the recipe has actually moved.
    declared = next(
        (s for s in ds.sources.of_kind("tracks") if s.id == "zfish"), None
    )
    if declared is not None and declared.path != str(src_dir):
        ds.remove_scan_source("tracks", "zfish")
        declared = None
    if declared is None:
        ds.add_scan_source(
            TracksScanSource(
                id="zfish",
                path=str(src_dir),
                patterns=("*.mat",),
                src_format="zfish_trx_mat",
            )
        )
    ds.scan_tracks()
    seen, indexed = len(list(src_dir.rglob("*.mat"))), len(ds.read_tracks_raw_index())
    print(
        f"a naive rglob finds {seen} files; the scan indexed {indexed}"
        + (
            f"  ({seen - indexed} AppleDouble sidecars dropped)"
            if seen != indexed
            else "  (no sidecars on this volume)"
        )
    )
    outcome = ds.convert_all_tracks(params={"strict_schema": True}, overwrite=RESET)
    assert outcome.failed == 0, "a trial failed to convert"
    return outcome


if SOURCE == "convert":
    if RESET and WORK.exists():
        shutil.rmtree(WORK)
        WORK.mkdir(parents=True)
    ds = make_dataset(WORK, "zebrafish WT collective motion")
    print(convert_into(ds, SRC))
else:
    # WORK already holds all 33 converted tables. Run the converter above on the
    # two-trial sample into a scratch dataset and check it reproduces the shipped
    # tables exactly -- so the converter printed in this notebook is demonstrably
    # the one that produced the data the rest of it reads, not a description of it.
    ds = make_dataset(WORK, "zebrafish WT collective motion")
    scratch = Path(tempfile.mkdtemp(prefix="zfish-verify-"))
    verify_ds = make_dataset(scratch, "zebrafish converter check")
    convert_into(verify_ds, SRC)
    for _, row in read_tracks_index(verify_ds).iterrows():
        group, sequence = str(row["group"]), str(row["sequence"])
        ours = verify_ds.load_tracks(group, sequence).reset_index(drop=True)
        shipped = ds.load_tracks(group, sequence).reset_index(drop=True)
        pd.testing.assert_frame_equal(ours, shipped, check_exact=True)
        print(
            f"converter check: {sequence} reproduces the shipped table exactly "
            f"({len(ours):,} rows, {len(ours.columns)} columns)"
        )
    shutil.rmtree(scratch, ignore_errors=True)

index = read_tracks_index(ds)
entries = [(str(r["group"]), str(r["sequence"])) for _, r in index.iterrows()]
# n_rows is a text column, as every cell of a typed IndexCSV is on the way out.
# Summing it without the conversion concatenates instead of adding.
n_rows = pd.to_numeric(index.n_rows)
print(
    f"\n{len(index)} entries, {int(n_rows.sum()):,} rows, "
    f"schema {sorted(index.std_format.unique())}, "
    f"variant {sorted(index.run_id.unique())}"
)


def read_feature(result, columns=None):
    """Concatenate a feature run's per-sequence parquet outputs."""
    run_dir = ds.get_root("features") / result.feature / result.run_id
    return pd.concat(
        [
            pd.read_parquet(p, columns=columns)
            for p in sorted(run_dir.glob("*.parquet"))
        ],
        ignore_index=True,
    )


def fold_feature(result, fn, columns=None):
    """Reduce a feature run one parquet at a time.

    The per-frame features emit one row per individual per frame -- about 21
    million over 33 trials, several gigabytes concatenated, for summaries that
    are a few dozen numbers. Folding keeps one trial in memory at a time.
    """
    run_dir = ds.get_root("features") / result.feature / result.run_id
    return [
        fn(pd.read_parquet(p, columns=columns))
        for p in sorted(run_dir.glob("*.parquet"))
    ]

## 5. Check the converter worked

A converter that transposes an axis or flips a sign produces a table that passes
every schema check and is entirely wrong. Two cheap checks catch most of it:

1. **`ANGLE` agrees with the direction of travel.** This is the check that
   earns the schema exception, and with no keypoints there is nothing else in
   the table to test the heading against -- so test it against the fish's own
   motion. A y-flipped image frame would push the disagreement to about 90
   degrees and a tail-resolved angle to 180, so both failures are unmistakable.
   Note this does not make `ANGLE` a velocity estimate: a few percent of moving
   frames sit well away from the travel direction, which is what fish do when
   they turn or glide.
2. **Missing data went where it should.** Rows dropped for a missing position,
   rows kept with a missing heading, and rows kept with no usable ellipse are
   three different things, counted separately.

Run the equivalent on your own converter and you will catch most of what can go
wrong silently.

In [ ]:
def wrap(v):
    """Angles to (-pi, pi]. Every angular comparison below is circular."""
    return np.arctan2(np.sin(v), np.cos(v))


# Only over the trials cell 3 actually surveyed. In download mode `survey` covers
# the two-trial sample while `entries` covers all 33 shipped tables, and subtracting
# one from the other is how the "dropped" line below becomes a large negative number.
counted = [(g, s) for g, s in entries if s in set(survey.trial)]
assert counted, "no surveyed trial is in the tracks index"
counted_survey = survey[survey.trial.isin({s for _, s in counted})]

rows_kept = no_heading = no_ellipse = 0
for group, sequence in counted:
    t = ds.load_tracks(group, sequence)
    rows_kept += len(t)
    no_heading += int(t.ANGLE.isna().sum())
    no_ellipse += int(t.body_length.isna().sum())

# Heading against the direction of travel, on one trial. Steps are taken only
# between consecutive frames of one individual, and only where the fish moved
# far enough for its direction of travel to mean anything.
t0 = ds.load_tracks(*counted[0]).sort_values(["id", "frame"])
g = t0.groupby("id")
dx, dy, dframe = g.X.diff(), g.Y.diff(), g.frame.diff()
moving = (dframe == 1) & (np.hypot(dx, dy) > 3.0) & t0.ANGLE.notna()
travel = np.arctan2(dy[moving], dx[moving])
theta = t0.ANGLE[moving]
print(f"heading check on {counted[0][1]}: {int(moving.sum()):,} moving frames")
for label, candidate in (
    ("ANGLE", theta),
    ("-ANGLE  (y flipped)", -theta),
    ("ANGLE+pi (tail)   ", theta + np.pi),
):
    err = np.abs(wrap(candidate - travel))
    print(f"  median |{label:19s} - direction of travel| = {np.median(err):.3f} rad")
print(
    f"  {100 * np.mean(np.abs(wrap(theta - travel)) < np.deg2rad(30)):.1f}% "
    "of moving frames within 30 degrees"
)

raw_rows = int((counted_survey.fish * counted_survey.frames).sum())
print(f"\nover the {len(counted)} trial(s) surveyed above:")
print(f"rows in the raw files : {raw_rows:,}")
print(
    f"rows in the tables    : {rows_kept:,} "
    f"({raw_rows - rows_kept:,} dropped for a missing position)"
)
print(f"  of those, no heading: {no_heading:,} ({100 * no_heading / rows_kept:.2f}%)")
print(f"  of those, no ellipse: {no_ellipse:,} ({100 * no_ellipse / rows_kept:.2f}%)")

BL = float(pd.concat([ds.load_tracks(*e).body_length for e in counted[:3]]).median())
print(
    f"\nbody length {BL:.1f} px = {BL / PX_PER_CM:.2f} cm "
    "(median over fitted rows; the a=0 non-fits are NaN and drop out)"
)

## 6. Collective motion

`collective-motion-metrics` reduces each frame to a handful of group-level
numbers. Two order parameters, from Tunstrom et al. (2013), carry most of it.
With `u_i` the unit heading of individual `i` and `r_i` the unit vector from the
group centroid to it:

- **polarization** `O_p = |mean(u_i)|` -- how aligned the group is, 0 to 1.
- **rotation** `O_r = |mean(u_i x r_i)|` -- how much it circles its own center.

High polarization with low rotation is a **polarized** school; the reverse is a
**mill**; both low is a disordered **swarm**; anything else is **transitional**.
The feature emits that label as a `state` column.

Parameters worth setting:

- `fps` -- derive the time step from the frame column rather than the timestamps.
- `max_frame_gap=1` -- do not difference across a dropped frame. Without it a
  one-frame gap yields a velocity averaged over two intervals.
- `min_group_speed=10.0` px/s -- below this the centroid heading is not
  computed. A milling group's centroid barely moves, and `arctan2(0, 0)` returns
  due east without complaint.
- `heading_source="auto"` (the default) uses `ANGLE` where the table has a usable
  one and the direction of travel otherwise. The output records which, in a
  `heading_source` column, so assert on it rather than assuming.

This feature emits one row per frame, so do not run it with `overlap_frames`
above 0.

In [ ]:
from mosaic.behavior.feature_library.collective_motion_metrics import (
    CollectiveMotionMetrics,
)

cmm = ds.run_feature(
    CollectiveMotionMetrics(
        params={
            "fps": FPS,
            "max_frame_gap": 1,
            "min_group_speed": 10.0,
        }
    ),
    parallel_workers=WORKERS,
    parallel_mode="process",
)
sources = read_feature(cmm, columns=["heading_source"]).heading_source.unique()
assert list(sources) == ["orientation"], f"expected measured headings, got {sources}"
print("heading_source:", sources[0], "-- ANGLE was used, not a velocity fallback")

## 7. Trajectory smoothing and interpolation

The raw tracks contain occasional jumps that are not swimming: the 99.9th
percentile of frame-to-frame speed is several times any real burst, and the
maximum is far beyond what a 2 cm fish can do. Anything that differences
positions in time is exposed to them, which is exactly what the force maps in
section 9 do. The order parameters above are not, because they read `ANGLE`.

`trajectory-smooth` repairs those frames:

- `speed_threshold` (px/s) flags frames whose step implies an impossible speed.
  2000 px/s is about 50 cm/s here, above any real burst and far below the jumps.
- `expand_frames` widens each flag, since a bad position corrupts the steps on
  both sides of it.
- `interpolate_centroid=True` fills the flagged gaps in `X`/`Y` by interpolation.
  `interpolate_pose` is left off -- the pose here is a derived ellipse axis, not
  a measurement to reconstruct.
- The output carries a `bad_frame` column marking what was repaired, so
  downstream features can exclude those rows.

`speed-angvel` then derives speed from the smoothed positions. It is a separate
feature because speed is derived, and `mosaic_v1` forbids a tracker-written
`SPEED`. **`smooth_window` is what creates the `speed_smooth` column** -- without
it, only `speed` exists, and the force chain below asks for `speed_smooth` by
name.

In [ ]:
from mosaic.behavior.feature_library.speed_angvel import SpeedAngvel
from mosaic.behavior.feature_library.trajectory_smooth import TrajectorySmooth
from mosaic.core.pipeline.types import Inputs

smooth = ds.run_feature(
    TrajectorySmooth(
        params={
            "speed_threshold": 2000.0,
            "fps": FPS,
            "interpolate_centroid": True,
            "interpolate_pose": False,
            "expand_frames": 8,
            "savgol_window": None,
            "savgol_polyorder": 1,
        }
    ),
    parallel_workers=WORKERS,
    parallel_mode="process",
)

speed = ds.run_feature(
    SpeedAngvel(
        Inputs((smooth,)), params={"step_size": 4, "smooth_window": 5, "fps": FPS}
    ),
    parallel_workers=WORKERS,
    parallel_mode="process",
)

bad = np.mean(
    fold_feature(smooth, lambda d: float(d.bad_frame.mean()), columns=["bad_frame"])
)
print(f"frames repaired by trajectory-smooth: {100 * bad:.2f}%")

## 8. Nearest neighbor

`nearest-neighbor` takes no parameters. For each fish in each frame it finds the
closest other fish and reports the offset in the world frame and rotated into
the focal fish's own frame -- the coordinate system the force maps are drawn in.

It runs twice here, and the two runs answer different questions. On the **raw**
tracks it measures the neighbor geometry the tracker actually reported, which is
what section 12 summarizes. On the **smoothed** tracks it feeds the force chain,
so that neighbor position and the response it explains come from one coordinate
series rather than two.

Check `nn_ego_unrotated` on the output. It is `True` when the input table had no
heading column, in which case the `_ego` offsets are the *world* offsets under
an egocentric name -- which looks like a perfectly good measurement and is not.

One behavior to be aware of: the neighbor is chosen with `np.argmin` over a row
of distances, and `np.argmin` returns the index of a NaN when one is present, so
a position-less fish would become everyone's nearest neighbor for that frame.
The converter's `drop_undetected` prevents it, because such a row is not in the
table. Section 12 bounds what is left.

In [ ]:
from mosaic.behavior.feature_library.nearestneighbor import NearestNeighbor

nn_raw = ds.run_feature(
    NearestNeighbor(), parallel_workers=WORKERS, parallel_mode="process"
)
nn = ds.run_feature(
    NearestNeighbor(Inputs((smooth,))),
    parallel_workers=WORKERS,
    parallel_mode="process",
)
print("nearest-neighbor run on the raw tracks and on the smoothed tracks")

## 9. The social-force chain

A force map asks: given that my nearest neighbor is *there*, in my own body
frame, what do I do next? The response is measured a fixed number of frames
later, as a change in heading (the **turn** response) and a change in speed (the
**speed** response). Binned by neighbor position over millions of frames, the
result is a map of the effective interaction.

Two features do it, and one supporting one:

| Step | What it does |
| --- | --- |
| `id-tag-columns` | Attaches a per-individual category column, here a `Focal_fish` flag |
| `nn-delta-response` | Pairs each neighbor position with the response `diff_numframes` later |
| `nn-delta-bins` | Bins those pairs into the 2D map, as sums and counts |

Three parameters must be set explicitly, because each default fails quietly
here:

- **`speed_col="speed_smooth"`.** The default is `"SPEED"`, a column `mosaic_v1`
  forbids, so it can never exist in these tables.
- **`sampling.fps_default=FPS`.** It defaults to 30.0 and scales the turn
  response directly into radians per second, so leaving it halves every number on
  a 60 Hz recording.
- **Run `id-tag-columns` even when every individual is focal.** `nn-delta-bins`
  emits zero rows when neither the focal nor the neighbor category column is
  present -- not an error, an empty result.

### Bin geometry

`binmax` sets how far the map reaches, `max_for_avg` the half-width of the band
used to collapse it into a 1D profile. Both are in the coordinate unit, so
**pixels** here; only the plot axes get converted to centimeters.

Set `max_for_avg` to the median neighbor distance and `binmax` to about three
times it. Anchoring `binmax` on a high percentile instead is tempting and wrong
for this data: the 95th percentile is roughly five times the median, because
fish leave the group and come back, so most of the grid would be spent on outer
cells holding almost no data.

`max_for_avg` affects only the feature's own 1D rows -- the 2D grid does not
depend on it -- so the band used at plot time can change without a recompute.
Changing `binmax` does re-run `nn-delta-bins`.

In [ ]:
from mosaic.behavior.feature_library.id_tag_columns import IdTagColumns
from mosaic.behavior.feature_library.nn_delta_bins import NearestNeighborDeltaBins
from mosaic.behavior.feature_library.nn_delta_response import NearestNeighborDelta

# Every fish is focal: the map pools all of them. The tags still have to exist.
focal = pd.DataFrame(
    [
        {"group": g, "sequence": s, "id": int(i), "focal": True}
        for g, s in entries
        for i in sorted(ds.load_tracks(g, s).id.unique())
    ]
)
focal_csv = WORK / "focal_tags.csv"
focal.to_csv(focal_csv, index=False)
ds.convert_id_tags_from_csv(
    csv_path=focal_csv, csv_type="multi", field_columns=["focal"], overwrite=True
)
tags = ds.run_feature(
    IdTagColumns(
        params={
            "label_kind": "id_tags",
            "fields": ["focal"],
            "field_renames": {"focal": "Focal_fish"},
        }
    )
)

nn_dist = pd.concat(
    fold_feature(nn, lambda d: d.nn_dist.dropna(), columns=["nn_dist"]),
    ignore_index=True,
)
p50 = float(nn_dist.quantile(0.50))
NBINS = 45
BINMAX = float(round(3.0 * p50))
MAX_FOR_AVG = float(round(p50))
print(f"median neighbor distance {p50:.1f} px = {p50 / PX_PER_CM:.2f} cm")
print(
    f"binmax {BINMAX:.0f} px = {BINMAX / PX_PER_CM:.2f} cm, "
    f"max_for_avg {MAX_FOR_AVG:.0f} px = {MAX_FOR_AVG / PX_PER_CM:.2f} cm"
)

delta = ds.run_feature(
    NearestNeighborDelta(
        Inputs((smooth, nn, speed, tags)),
        params={
            "speed_col": "speed_smooth",
            "diff_numframes": DIFF_NUMFRAMES,
            "sampling": {"fps_default": FPS},
            "tag_cols": ["bad_frame"],
        },
    ),
    parallel_workers=WORKERS,
    parallel_mode="process",
    filter_start_frame=ACCLIMATION_FRAMES,
)

bins = ds.run_feature(
    NearestNeighborDeltaBins(
        Inputs((delta, tags)),
        params={
            "nbins": NBINS,
            "binmax": BINMAX,
            "max_for_avg": MAX_FOR_AVG,
            "antisymm": True,
            "exp_col": "group",
            "trial_col": "sequence",
            "focal_category_col": "Focal_fish",
            "neighbor_category_col": "neighbor_focal",
            "exclude_cols": ["bad_frame", "neighbor_bad_frame"],
        },
    ),
    parallel_workers=1,
    parallel_mode="thread",
)

BINS2D = read_feature(bins).query("dim == '2d'")
populated = int((BINS2D["count"] > 0).sum())
assert populated > 0, "empty force map -- did the focal-tag cell run?"
print(f"{populated:,} populated bins over {BINS2D.sequence.nunique()} trials")

# Part B. Loading results and plotting

Nothing below calls into mosaic. Every cell reads results written in Part A and
draws them, so this is the part you would replace wholesale with your own
plotting.

## 10. Tracks and the arena

One frame, showing what the converter produced: a centroid and a measured
heading per fish, which is all this tracker gives.

The right panel fits the arena to the trajectories. A bounding box is the wrong
shape for a circle -- the extreme x and y are four points that need not be
diametrically opposite, and the circle they imply leaves several percent of the
data outside it. `fit_arena` instead takes the outermost point in each angular
sector, the fish that were against the wall all the way around, and fits a
circle to those by least squares. Worth having because the `arena` field in the
file is stale template data, so the geometry has to come from the tracks.

In [ ]:
def fit_arena(x, y, n_sectors=180, edge_q=0.995, cover_q=0.999, iters=4):
    """Center and radius of a circular arena, measured from the trajectories.

    Takes the outermost points in each angular sector -- the fish that were
    against the wall, all the way around -- and fits a circle to them by least
    squares, re-centering as it goes. The radius then covers `cover_q` of the
    data from that center, so the circle bounds the occupied region rather than
    cutting through it.
    """
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x)[ok], np.asarray(y)[ok]
    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    for _ in range(iters):
        ang = np.arctan2(y - cy, x - cx)
        r = np.hypot(x - cx, y - cy)
        sector = np.clip(
            ((ang + np.pi) / (2 * np.pi) * n_sectors).astype(int), 0, n_sectors - 1
        )
        edge = [
            np.where((sector == k) & (r >= np.quantile(r[sector == k], edge_q)))[0]
            for k in range(n_sectors)
            if (sector == k).sum() > 50
        ]
        px, py = x[np.concatenate(edge)], y[np.concatenate(edge)]
        # Algebraic circle fit: x^2 + y^2 = 2a x + 2b y + c, linear in (a, b, c).
        sol, *_ = np.linalg.lstsq(
            np.c_[2 * px, 2 * py, np.ones(len(px))], px**2 + py**2, rcond=None
        )
        cx, cy = float(sol[0]), float(sol[1])
    return cx, cy, float(np.quantile(np.hypot(x - cx, y - cy), cover_q))


tracks = ds.load_tracks(*entries[0])
frame_no = int(tracks.frame.quantile(0.1, interpolation="nearest"))
snap = tracks[tracks.frame == frame_no]
cx, cy, arena_r = fit_arena(tracks.X.to_numpy(), tracks.Y.to_numpy())
outside = float((np.hypot(tracks.X - cx, tracks.Y - cy) > arena_r).mean())

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 5.2))

# Zoomed to the fish: six 2 cm fish in a 47 cm arena are a few pixels apart at
# arena scale, and this panel is about the headings.
pad = 8 * BL
ax.quiver(
    snap.X,
    snap.Y,
    np.cos(snap.ANGLE),
    np.sin(snap.ANGLE),
    angles="xy",
    scale_units="xy",
    scale=1 / (2.0 * BL),
    width=0.007,
    color="#333",
    zorder=4,
)
ax.plot(snap.X, snap.Y, "o", ms=7, color="#d9480f", zorder=5)
ax.set(
    title=f"{entries[0][1]}\nframe {frame_no}, n={len(snap)}",
    aspect="equal",
    xlim=(snap.X.min() - pad, snap.X.max() + pad),
    ylim=(snap.Y.min() - pad, snap.Y.max() + pad),
    xticks=[],
    yticks=[],
)
ax.invert_yaxis()  # image coordinates: +Y points down

ax2.plot(tracks.X[::200], tracks.Y[::200], ",", color="#3b7dd8", alpha=0.5)
ax2.add_patch(
    plt.Circle(
        (cx, cy),
        arena_r,
        fill=False,
        color="#c9541a",
        lw=1.5,
        label=f"fitted arena, r = {arena_r / PX_PER_CM:.1f} cm",
    )
)
ax2.plot([cx], [cy], "+", color="#c9541a", ms=10, mew=1.5)
ax2.set(
    title="where the fish are, and the arena fitted to them",
    aspect="equal",
    xticks=[],
    yticks=[],
)
ax2.legend(fontsize=8, loc="upper right")
ax2.invert_yaxis()
fig.suptitle(
    "centroid (orange) and measured heading (grey) -- this tracker reports "
    "no keypoints",
    fontsize=10,
)
fig.tight_layout()
print(
    f"arena center ({cx:.0f}, {cy:.0f}) px, radius {arena_r:.0f} px "
    f"= {arena_r / PX_PER_CM:.2f} cm, so {2 * arena_r / PX_PER_CM:.1f} cm across; "
    f"{100 * outside:.2f}% of positions fall outside it"
)

## 11. Collective motion

The order parameters over time for one trial, the fraction of frames in each
collective state across all of them, and the joint distribution the state
definitions cut up.

State fractions come from the per-frame classification as the feature emits it,
with no smoothing. With six fish the per-frame order parameters are noisy --
their variance goes as 1/N -- so read the transitional fraction as a statement
about frames rather than about how long the group holds a recognizable
formation. The time series below shows three minutes at full resolution rather
than the whole half hour, because 108,000 points across a plot is solid ink and
smoothing it would hide exactly the variability worth seeing.

In [ ]:
from mosaic.behavior.feature_library.collective_math import STATE_HIGH, STATE_LOW

metrics = read_feature(
    cmm,
    columns=[
        "group",
        "sequence",
        "frame",
        "time",
        "polarization",
        "rotation",
        "state",
        "n_ids",
    ],
)

STATES = ["Polarized", "Milling", "Swarm", "Transitional", "Undefined"]
COLORS = {
    "Polarized": "#1f6fb4",
    "Milling": "#c9541a",
    "Swarm": "#4c9a5a",
    "Transitional": "#9b8ea3",
    "Undefined": "#cccccc",
}

fractions = (
    (metrics.state.value_counts(normalize=True) * 100).reindex(STATES).fillna(0.0)
)
per_trial = (
    (
        metrics.groupby("sequence")
        .state.value_counts(normalize=True)
        .unstack(fill_value=0)
        * 100
    )
    .reindex(columns=STATES)
    .fillna(0.0)
)
print(f"{len(metrics):,} frames over {metrics.sequence.nunique()} trials")
display(fractions.round(2).to_frame("% of frames"))

fig, (ax, ax2) = plt.subplots(
    2, 1, figsize=(11, 6.4), gridspec_kw={"height_ratios": [2.2, 1]}
)
one = metrics[metrics.sequence == metrics.sequence.iloc[0]].sort_values("time")
# A three-minute window at full resolution, not the whole half hour. Drawing
# 108,000 points across a thousand pixels is solid ink whatever the line width,
# and smoothing to fix that would hide how noisy these quantities really are
# with six fish -- which is the thing worth seeing here.
WINDOW_S = 180
win = one[one.time <= one.time.min() + WINDOW_S]
ax.plot(
    win.time - win.time.min(), win.polarization, lw=0.6, color="#1f6fb4", label="$O_p$"
)
ax.plot(win.time - win.time.min(), win.rotation, lw=0.6, color="#c9541a", label="$O_r$")
for level in (STATE_LOW, STATE_HIGH):
    ax.axhline(level, color="#666", lw=0.7, ls="--")
ax.set(
    xlabel="time (s)",
    ylabel="order parameter",
    ylim=(0, 1.02),
    xlim=(0, WINDOW_S),
    title=f"{one.sequence.iloc[0]}, first {WINDOW_S} s at full 60 Hz resolution",
)
ax.legend(loc="upper right", ncols=2, fontsize=8, framealpha=0.9)

left = 0.0
for state in [s for s in STATES if fractions[s] > 0.01]:
    ax2.barh(
        0, fractions[state], left=left, color=COLORS[state], label=state, height=0.5
    )
    pts = per_trial[state].to_numpy()
    ax2.plot(left + pts, np.zeros_like(pts), "|", color="k", ms=14, mew=1.0, alpha=0.5)
    left += fractions[state]
ax2.set(
    yticks=[],
    xlabel="% of frames",
    xlim=(0, 100),
    title=f"collective state over {len(per_trial)} trials "
    "(tick marks are individual trials)",
)
ax2.legend(ncols=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.45))
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.8))
sub = metrics.dropna(subset=["rotation", "polarization"])
h = ax.hist2d(
    sub.rotation,
    sub.polarization,
    bins=60,
    range=[[0, 1], [0, 1]],
    cmap="magma",
    cmin=1,
    density=True,
)
for level in (STATE_LOW, STATE_HIGH):
    ax.axhline(level, color="w", lw=0.8, ls="--", alpha=0.7)
    ax.axvline(level, color="w", lw=0.8, ls="--", alpha=0.7)
ax.set(
    xlabel="rotation $O_r$",
    ylabel="polarization $O_p$",
    aspect="equal",
    title="collective state space",
)
fig.colorbar(h[3], ax=ax, shrink=0.82)
fig.tight_layout()

## 12. Neighbor distance

Nearest-neighbor distance is a comparable number in its own right -- it is one
of the quantities the paper this data comes from reports per line. Summarized
per trial from the **raw**-track run, so it describes the tracker's own output
rather than the smoothed version.

Two filters are applied and both are worth stating. Distances of exactly zero
are two fish reported at one pixel, which is the tracker briefly losing them
into each other rather than a measurement of proximity; they are counted and
excluded. And the median is recomputed over only those frames holding all six
fish, which bounds the bias from dropping position-less rows.

In [ ]:
SUBSAMPLE = 20_000
NN_COLS = ["frame", "sequence", "nn_dist", "nn_ego_unrotated"]


def nn_summary(d):
    assert not d.nn_ego_unrotated.any(), (
        "nn_ego_unrotated is True: no ANGLE column reached the feature, so the "
        "'_ego' offsets are world-frame under an egocentric name"
    )
    dist = d.nn_dist.dropna()
    coincident = int((dist == 0).sum())
    dist = dist[dist > 0]
    # The same exclusion on both sides, so the comparison isolates the effect of
    # a missing fish rather than mixing in the coincident-pair filter.
    full6 = (d.groupby("frame").nn_dist.transform("size") == 6) & (d.nn_dist > 0)
    return {
        "sequence": d.sequence.iloc[0],
        "n": int(len(dist)),
        "coincident": coincident,
        "mean_px": float(dist.mean()),
        "median_px": float(dist.median()),
        "sd_px": float(dist.std()),
        "median_full_px": float(d.nn_dist[full6].median()),
        # A seeded subsample: a kernel density over all 21 million rows resolves
        # nothing the plot can show.
        "sample": dist.sample(n=min(SUBSAMPLE, len(dist)), random_state=0).to_numpy(),
    }


def short_label(seq):
    """`data09092015_10_26_AMAB` -> `2015-09-09 10:26a`, for a readable axis."""
    m = re.match(r"data(\d{2})(\d{2})(\d{4})_(\d{2})_(\d{2})_(AM|PM)", seq)
    if not m:
        return seq[:16]
    mm, dd, yyyy, hh, mi, ampm = m.groups()
    return f"{yyyy}-{mm}-{dd} {hh}:{mi}{ampm[0].lower()}"


parts = fold_feature(nn_raw, nn_summary, columns=NN_COLS)
samples = {p["sequence"]: p["sample"] / PX_PER_CM for p in parts}
nn_trials = pd.DataFrame([{k: v for k, v in p.items() if k != "sample"} for p in parts])
for src, dst in (
    ("mean_px", "mean_cm"),
    ("median_px", "median_cm"),
    ("sd_px", "sd_cm"),
):
    nn_trials[dst] = nn_trials[src] / PX_PER_CM
nn_trials["mean_bl"] = nn_trials.mean_px / BL
nn_trials = nn_trials.sort_values("median_cm").reset_index(drop=True)

print(f"nearest-neighbor distance, per trial ({len(nn_trials)} trials)")
print(
    f"  mean of the per-trial means : {nn_trials.mean_cm.mean():.2f} cm "
    f"({nn_trials.mean_bl.mean():.2f} body lengths)"
)
print(
    f"  spread across trials        : {nn_trials.mean_cm.std():.2f} cm sd, "
    f"range {nn_trials.mean_cm.min():.2f} to {nn_trials.mean_cm.max():.2f}"
)
coin = int(nn_trials.coincident.sum())
print(
    f"  coincident pairs excluded   : {coin:,} rows "
    f"({100 * coin / (coin + nn_trials.n.sum()):.2f}%), two fish at one pixel"
)
shift = (nn_trials.median_full_px - nn_trials.median_px).abs().max()
print(
    f"  largest shift when restricted to frames holding all six fish: "
    f"{shift:.2f} px ({shift / PX_PER_CM:.3f} cm)\n"
)

display(
    nn_trials[
        ["sequence", "n", "coincident", "mean_cm", "median_cm", "sd_cm", "mean_bl"]
    ].round({"mean_cm": 2, "median_cm": 2, "sd_cm": 2, "mean_bl": 2})
)

# Violins are drawn on log10(distance): the distribution spans a factor of
# twenty, a median near 2 cm against a tail past 40, and on a linear axis every
# trial collapses into the bottom tenth of the panel.
data = [np.log10(samples[s]) for s in nn_trials.sequence]
pos = np.arange(1, len(data) + 1)

fig, ax = plt.subplots(figsize=(max(9.0, 0.40 * len(data)), 5.4))
v = ax.violinplot(data, positions=pos, widths=0.85, showmedians=True, showextrema=False)
for body in v["bodies"]:
    body.set_facecolor("#1f6fb4")
    body.set_alpha(0.55)
    body.set_edgecolor("none")
v["cmedians"].set_color("#123")
v["cmedians"].set_linewidth(1.0)
ax.plot(
    pos,
    np.log10(nn_trials.mean_cm),
    "o",
    ms=3.6,
    color="#c9541a",
    zorder=5,
    label="trial mean",
)
ax.axhline(
    np.log10(nn_trials.median_cm.median()),
    color="#666",
    lw=0.9,
    ls="--",
    label=f"median of trial medians ({nn_trials.median_cm.median():.2f} cm)",
)

TICKS_CM = np.array([0.5, 1, 2, 5, 10, 20, 40])
ax.set_yticks(np.log10(TICKS_CM))
ax.set_yticklabels([f"{t:g}" for t in TICKS_CM])
ax.set_ylim(np.log10(0.3), np.log10(55))
ax.set_xticks(pos)
ax.set_xticklabels(
    [short_label(s) for s in nn_trials.sequence], rotation=90, fontsize=7
)
ax.set(
    ylabel="nearest-neighbor distance (cm, log scale)",
    title=f"nearest-neighbor distance by trial, sorted by median "
    f"({len(data)} trials, {SUBSAMPLE:,} samples each)",
)
bl_cm = BL / PX_PER_CM
sec = ax.secondary_yaxis("right")
sec.set_yticks(np.log10(TICKS_CM))
sec.set_yticklabels([f"{t / bl_cm:.1f}" for t in TICKS_CM])
sec.set_ylabel("body lengths")
ax.grid(axis="y", color="#ddd", lw=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()

## 13. Force maps

`nn-delta-bins` writes **sums and counts**, not means, so that results can be
pooled across trials and divided once at the end. `to_grid` below does that
division.

### The 2D grid and its two axes

The grid is not square. `neighbor_x` is binned against
`linspace(-binmax, binmax, nbins)` and `neighbor_y` against
`linspace(-binmax, binmax, nbins + 1)`, which is **44** and **45** bins
respectively -- the grid is `(nbins - 1, nbins)`.

That matters when collapsing to a 1D profile, because the two metrics come out
on different axes. **Turn** bands on `bin_x` and returns values over `bin_y`, 45
of them; **speed** bands on `bin_y` and returns values over `bin_x`, 44. Pairing
a profile with the wrong axis drops a bin and mislabels the spacing by about 2%,
without any error.

So `collapse_2d_to_1d` returns the bin centers alongside the values, and every
caller asserts that the two lengths match. If you write your own collapse, do
the same.

Two unit notes. `dspeed` is a plain difference of `speed_smooth` across the lag, not
divided by the time step -- only the turn response gets that treatment -- so it is a
speed, not an acceleration.

And it arrives in **pixels per second**, because `speed-angvel` derives it from `X`/`Y`
and those are video pixels. The plotting helper's `scale` argument converts the bin
*centres* to centimetres, which is the axes; the binned response values need the same
division before they can carry a cm/s label. `VALUE_SCALE` below is that division, and
it is 1 for the turn response, which is rad/s and has no length in it.

In [ ]:
def bin_centers(nbins, binmax):
    """The two axes of the 2D grid, which are not the same length."""
    x_edges = np.linspace(-binmax, binmax, nbins)
    y_edges = np.linspace(-binmax, binmax, nbins + 1)
    return (0.5 * (x_edges[:-1] + x_edges[1:]), 0.5 * (y_edges[:-1] + y_edges[1:]))


def select_2d(bins2d, metric, neighbor_cat="all"):
    sub = bins2d[bins2d["metric"] == metric]
    return sub[sub["neighbor_category"].astype(str) == str(neighbor_cat)]


def to_grid(sub, nbins):
    """Pool (bin_x, bin_y, sum_value, count) rows into a mean grid."""
    if sub.empty:
        return np.full((nbins - 1, nbins), np.nan)
    ix, iy = sub["bin_x"].to_numpy(int), sub["bin_y"].to_numpy(int)
    sums, cnts = sub["sum_value"].to_numpy(float), sub["count"].to_numpy(float)
    m = np.isfinite(sums) & np.isfinite(cnts)
    flat, size = ix[m] * nbins + iy[m], (nbins - 1) * nbins
    gs = np.bincount(flat, weights=sums[m], minlength=size).reshape(nbins - 1, nbins)
    gc = np.bincount(flat, weights=cnts[m], minlength=size).reshape(nbins - 1, nbins)
    with np.errstate(divide="ignore", invalid="ignore"):
        grid = gs / gc
    grid[gc == 0] = np.nan
    return grid


def collapse_2d_to_1d(bins2d, metric, nbins, binmax, band, trial_col="sequence"):
    """Marginalize the 2D map over a band on one axis -> one profile per trial.

    Returns `(wide, centers)`. The centers come back with the values because the
    two axes have different lengths, so the caller cannot pair them wrongly.
    """
    x_centers, y_centers = bin_centers(nbins, binmax)
    sub = select_2d(bins2d, metric)
    sub = sub[sub["bin_x"].notna() & sub["bin_y"].notna()]
    if sub.empty:
        return pd.DataFrame(), np.array([])

    if metric == "turn":
        band_col, band_centers, result_col, centers = (
            "bin_x",
            x_centers,
            "bin_y",
            y_centers,
        )
    else:
        band_col, band_centers, result_col, centers = (
            "bin_y",
            y_centers,
            "bin_x",
            x_centers,
        )

    idx = sub[band_col].astype(int).to_numpy()
    inside = (idx >= 0) & (idx < len(band_centers))
    phys = np.where(
        inside, band_centers[np.clip(idx, 0, len(band_centers) - 1)], np.nan
    )
    sub = sub[(phys >= band[0]) & (phys <= band[1])]

    agg = (
        sub.groupby([trial_col, result_col], dropna=False)
        .agg(sum_value=("sum_value", "sum"), count=("count", "sum"))
        .reset_index()
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        agg["value"] = agg["sum_value"] / agg["count"]
    agg.loc[agg["count"] == 0, "value"] = np.nan
    # A pivot, not an enumerate: a trial missing a result bin must leave a hole,
    # not shift every later bin one place left.
    wide = agg.pivot_table(index=trial_col, columns=result_col, values="value").reindex(
        columns=range(len(centers))
    )
    return wide, centers


def bootstrap_ci(wide, n_boot=400, seed=0):
    """Mean profile and a percentile interval, resampling over trials."""
    rng = np.random.default_rng(seed)
    arr = wide.to_numpy(float)
    draws = np.array(
        [
            np.nanmean(arr[rng.integers(0, len(arr), len(arr))], axis=0)
            for _ in range(n_boot)
        ]
    )
    return (
        np.nanmean(arr, axis=0),
        np.nanpercentile(draws, 2.5, axis=0),
        np.nanpercentile(draws, 97.5, axis=0),
    )


def plot_force_map(grid, nbins, binmax, ax, metric, band, scale, lim=None, zlim=None):
    """Heatmap of a force map, focal fish at the origin facing up.

    `pcolormesh` gets the y centers as its horizontal coordinate and the x
    centers as its vertical one, so the display is transposed relative to the
    data: horizontal is the neighbor's left/right offset, vertical is
    ahead/behind.
    """
    x_c, y_c = (c / scale for c in bin_centers(nbins, binmax))
    lim = lim if lim is not None else 0.7 * binmax / scale
    xs = slice(*np.searchsorted(x_c, [-lim, lim]))
    ys = slice(*np.searchsorted(y_c, [-lim, lim]))
    if zlim is None:
        near = grid[np.abs(x_c) < lim][:, np.abs(y_c) < lim]
        zlim = (
            float(np.nanpercentile(np.abs(near), 97))
            if np.isfinite(near).any()
            else 1.0
        )
        zlim = zlim or 1.0
    mesh = ax.pcolormesh(
        y_c[ys],
        x_c[xs],
        grid[xs, ys],
        cmap="seismic",
        vmin=-zlim,
        vmax=zlim,
        rasterized=True,
    )
    ax.arrow(0, 0, 0, 0.18 * lim, head_width=0.06 * lim, color="k", zorder=5)
    span = ax.axhspan if metric == "turn" else ax.axvspan
    line = ax.axhline if metric == "turn" else ax.axvline
    lo, hi = (b / scale for b in band)
    span(lo, hi, color="lime", alpha=0.10, zorder=4)
    for v in (lo, hi):
        line(v, color="lime", lw=1.1, ls="--", alpha=0.8, zorder=4)
    ax.set_aspect("equal")
    return mesh


# The response VALUES need converting too, not just the axes. `speed_smooth` is
# derived from X/Y, which are video pixels, so `dspeed` is px/s -- dividing only the
# bin centres (as `plot_force_map`'s `scale` does) leaves the colorbar 40x too large
# under a cm/s label. The turn response is rad/s and carries no length unit, so it
# is divided by 1.
VALUE_SCALE = {"turn": 1.0, "speed": PX_PER_CM}

BAND = {"turn": (0.0, MAX_FOR_AVG), "speed": (-MAX_FOR_AVG / 2, MAX_FOR_AVG / 2)}
LABEL = {
    "turn": ("turn response", "$\\Delta\\theta$ (rad/s)"),
    "speed": ("speed response", "$\\Delta$ speed (cm/s)"),
}
AXIS = {"turn": "neighbor left / right (cm)", "speed": "neighbor ahead / behind (cm)"}
print("helpers defined")

In [ ]:
fig, axes = plt.subplots(
    2, 2, figsize=(10.5, 8.6), gridspec_kw={"height_ratios": [1.35, 1]}
)
for col, metric in enumerate(("turn", "speed")):
    band = BAND[metric]
    title, ylab = LABEL[metric]

    grid = to_grid(select_2d(BINS2D, metric), NBINS) / VALUE_SCALE[metric]
    ax = axes[0, col]
    mesh = plot_force_map(grid, NBINS, BINMAX, ax, metric, band, scale=PX_PER_CM)
    fig.colorbar(mesh, ax=ax, shrink=0.72, pad=0.03, label=ylab)
    ax.set(
        title=title,
        xlabel="neighbor left / right (cm)",
        ylabel="neighbor ahead / behind (cm)" if col == 0 else "",
    )

    wide, centers = collapse_2d_to_1d(BINS2D, metric, NBINS, BINMAX, band)
    assert wide.shape[1] == len(centers), (
        f"{metric}: {wide.shape[1]} values against {len(centers)} centers -- "
        "the 1D pairing is wrong"
    )
    mean, lo, hi = (v / VALUE_SCALE[metric] for v in bootstrap_ci(wide))
    x = centers / PX_PER_CM
    ax2 = axes[1, col]
    ax2.plot(x, mean, lw=1.6, color="#1f6fb4")
    ax2.fill_between(x, lo, hi, color="#1f6fb4", alpha=0.15)
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    ax2.set(
        xlim=(-0.7 * BINMAX / PX_PER_CM, 0.7 * BINMAX / PX_PER_CM),
        xlabel=AXIS[metric],
        ylabel=ylab,
        title=f"collapsed over the band, {len(wide)} trials, 95% CI",
    )

fig.suptitle(
    f"social force maps: response {DIFF_NUMFRAMES} frames "
    f"({1000 * DIFF_NUMFRAMES / FPS:.0f} ms) after the neighbor was there",
    fontsize=12,
)
fig.tight_layout()

### Reading these

The focal fish is at the origin of every panel, facing **up** (the black arrow).
The horizontal axis is where its nearest neighbor is, left or right; the vertical
axis is ahead or behind. The lime band marks the slice collapsed into the profile
below.

The **turn** map is antisymmetric by construction: `antisymm=True` mirrors each
sample in `y` and negates the response, imposing the left/right symmetry that
six fish over 33 trials would otherwise only approximate. What it does not
impose is the sign, or whether the sign changes with distance -- a turn toward
the neighbor at every distance is one interaction, a turn away at short range
and toward at long range is a different one. On this data it is the first: within
the roughly two body lengths the map spans, the response is toward the neighbor
throughout. That bounds rather than excludes a repulsion zone, which if present
sits closer than these bins resolve.

The **speed** map is duplicated in `y` without negation, since a neighbor to the
left and one to the right should affect speed the same way. It carries the
clearer signal here: a fish slows when a neighbor is ahead of it and speeds up
when one is behind, which is position-matching along the body axis.

One caution on interpretation. **This is a response to the nearest neighbor, not
a pairwise force law.** Each sample is conditioned on where the closest fish is
while the focal fish is responding to all five, and to the wall. When the nearest
neighbor is far the others are farther still, so the map at long range partly
reports "the group is over there". Recovering a pairwise rule needs a model.

## 14. Using this for your own data

Sections 1 to 3 are format-specific. Everything from section 4 onward is not --
it runs on any schema-valid track table, whatever produced it.

To adapt the converter, answer four questions:

1. **Does one file hold many sequences, or do many files make one sequence?**
   Set `enumerable = True` and implement `enumerate_sequences`, or
   `merges_per_sequence = True` and override `sequence_from_stem`. They are
   mutually exclusive. Here it was neither, which is the default.
2. **Does your tracker measure anything `mosaic_v1` forbids?** If it genuinely
   reports a heading or a speed, declare a schema that `extends="mosaic_v1"` and
   `allows` exactly that column, and be ready to show it is a measurement. If it
   does not, emit `mosaic_v1` and let a feature derive it.
3. **Does your tracker locate keypoints?** If it does not, write none.
   Keypoints are optional in `mosaic_v1`, and the features that need them refuse
   when they are absent, which is what you want. Do not copy the centroid into
   `poseX0`, and do not synthesize a body axis from a fitted ellipse: both look
   like located landmarks and neither is one. This converter took the second
   route at first, and the ellipse turned out to be missing on a fifth of rows
   and implausible on several percent more -- so the synthetic keypoints were
   unusable on a quarter of the data while looking perfectly well-formed.
4. **Are your coordinates already video pixels?** `mosaic_v1` says they are. If
   yours are physical units, convert in the converter and read the factor from
   the file rather than from a parameter.

And the one this dataset teaches most sharply: **check the tracker's metadata
against its own trajectories before believing any of it.** Six fields here are
stale, wrong, or a factor of two out, and each looks perfectly usable in
isolation.

[Adding a converter](../docs/guides/tracking/write-a-converter.md) covers the full contract,
including how to ship a converter inside mosaic rather than in a notebook, and
what the registry tests then check for you.